# Fast pending Transformer experiments
Paper config: 100k train, 2k val, 8k AdamW updates, seeds 2001–2005. Uses `fastexec.py` only as a validated faster backend. **Reversibility uses derangements on the reversible branch so η=0 exactly matches the paper's B.1 state-machine family.**

In [1]:
from pathlib import Path
from collections import Counter
import math, subprocess, numpy as np, pandas as pd, torch
import fastexec as fx
from compare_supervision import generate_unique
from src.registry import TASKS
from src.dataclass import Instance

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu"); SEEDS=tuple(range(2001,2006)); NTR,NVA=100_000,2_000
ARGS=fx.Cfg(steps=8_000,batch_size=128,lr=3e-4,weight_decay=0.0,grad_clip=1.0,eval_batch_size=256,compile="reduce-overhead",bf16=True)
CKPTS=(500,1000,2000,4000,8000); OUT=Path("results/pending_identifiability/paper"); OUT.mkdir(parents=True,exist_ok=True)
print("device=",DEVICE,"commit=",subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip()); print(ARGS)


device= cuda commit= bcca6cb1ff0c3f3afc7687e54fa6cc9f1328e4d4
Cfg(embedding=128, heads=4, layers=2, dropout=0.0, steps=8000, batch_size=128, lr=0.0003, weight_decay=0.0, grad_clip=1.0, batch_seed=12345, eval_batch_size=256, compile='reduce-overhead', bf16=True, log_every=50)


/home/aayus/Trace/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Reversibility dial — corrected state-machine intervention

In [2]:
K=16
def derange(rng):
    while True:
        p=rng.permutation(K)
        if np.all(p!=np.arange(K)): return p.astype(np.uint8)
def make_pool(n,D,seed):
    r=np.random.default_rng(seed); P=np.empty((n,2,K),np.uint8)
    for i in range(n):
        for j in range(2): P[i,j]=derange(r)
    return dict(perm=P,map=r.integers(0,K,(n,2,K),dtype=np.uint8),u=r.random((n,2)),s=r.integers(0,K,n,dtype=np.uint8),a=r.integers(0,2,(n,D),dtype=np.uint8))
def tables(p,i,eta): return np.where((p["u"][i]<eta)[:,None],p["map"][i],p["perm"][i]).astype(np.int64)
def rev_inst(p,i,eta):
    T=tables(p,i,eta); cur=int(p["s"][i]); acts=p["a"][i]; st=lambda x:f"{int(x):02d}"; ops="ab"
    prompt=f"a{''.join(st(x) for x in T[0])};b{''.join(st(x) for x in T[1])};s{st(cur)};u{''.join(ops[int(a)] for a in acts)}"; tr=[]
    for a in acts: a=int(a); nxt=int(T[a,cur]); tr.append(f"{st(cur)}{ops[a]}{st(nxt)}"); cur=nxt
    return Instance(prompt," ".join(tr),"",st(cur))
def build_rev(p,eta): return [rev_inst(p,i,eta) for i in range(len(p["s"]))]
def mu(p,eta):
    z=[]
    for i in range(len(p["s"])):
        T=tables(p,i,eta); s=np.arange(K)
        for a in p["a"][i]: s=T[int(a),s]
        q=np.bincount(s,minlength=K)/K; z.append(np.abs(q-1/K).max())
    return float(np.mean(z))
def run_rev():
    D=8; etas=(0,.1,.25,.5,1.0); trp,vap=make_pool(NTR,D,501),make_pool(NVA,D,101); task=TASKS["state_machine_8"]; path=OUT/"p3_reversibility_state_machine.csv"
    old=pd.read_csv(path) if path.exists() else pd.DataFrame(); rows=old.to_dict("records"); done=set() if old.empty else set(zip(old.eta_target.round(6),old["mode"],old.seed.astype(int)))
    for eta in etas:
        tr,va=build_rev(trp,eta),build_rev(vap,eta); counts=Counter(x.gold for x in va); maj=max(counts.values())/len(va); ent=-sum((c/len(va))*math.log(c/len(va)+1e-30) for c in counts.values()); m=mu(vap,eta); eff=float((vap["u"]<eta).mean()); print("eta",eta,"mu",m,"majority",maj)
        for mode in ("outcome","process"):
            split=fx.pack(tr,task,fx.TARGETS[mode],DEVICE)
            for seed in SEEDS:
                if (round(eta,6),mode,seed) in done: print("skip",eta,mode,seed); continue
                r=fx.run(task,tr,va,mode,seed,ARGS,DEVICE,condition=mode,split=split,target_of=fx.TARGETS[mode],desc=f"eta={eta:g}/{mode}/s{seed}")
                rows.append({"eta_target":eta,"eta_effective":eff,"mu":m,"majority_baseline":maj,"answer_entropy":ent,**r}); pd.DataFrame(rows).to_csv(path,index=False)
            del split; torch.cuda.empty_cache() if DEVICE.type=="cuda" else None
    df=pd.DataFrame(rows); agg=df.groupby(["eta_target","mode"],as_index=False).agg(mu=("mu","mean"),answer_accuracy=("answer_accuracy","mean"),answer_sd=("answer_accuracy","std"),majority_baseline=("majority_baseline","mean"),exact_trace_accuracy=("exact_trace_accuracy","mean"),n=("seed","count"))
    agg["excess_over_majority"]=agg.answer_accuracy-agg.majority_baseline; agg.to_csv(OUT/"p3_reversibility_state_machine_aggregate.csv",index=False); display(agg); return df,agg


In [ ]:
rev_df,rev_agg=run_rev()

eta 0 mu 0.0 majority 0.069


eta=0/outcome/s2001:   0%|          | 0/8000 [00:00<?, ?it/s]W0906 07:41:46.699000 213336 torch/_inductor/utils.py:1806] [0/0] Not enough SMs to use max_autotune_gemm mode


eta 0.1 mu 0.05865625 majority 0.0745


eta 0.25 mu 0.139125 majority 0.069


eta 0.5 mu 0.26559375 majority 0.07


eta=0.5/process/s2003:  32%|███▏      | 2562/8000 [00:18<00:46, 117.47it/s, loss=0.3510]

## 2. Full-Transformer supervision stride — qualitative monotone test

In [ ]:
def sparse_trace(inst,k): return " ".join(inst.correct_trace.split()[k-1::k])
def run_stride():
    D=16; ks=(1,2,4,8,16); task=TASKS["boolean_circuit_16"]; tr=generate_unique(task,NTR,501); va=generate_unique(task,NVA,101,{x.prompt for x in tr}); path=OUT/"p4a_stride_transformer.csv"
    old=pd.read_csv(path) if path.exists() else pd.DataFrame(); rows=old.to_dict("records"); done=set()
    if not old.empty:
        for (k,s),g in old.groupby(["k","seed"]):
            if g.step.max()>=max(CKPTS): done.add((int(k),int(s)))
    for k in ks:
        target=fx.stride_target(D,k); split=fx.pack(tr,task,target,DEVICE); mode="outcome" if k==D else "process"; gold=None if k==D else (lambda x,k=k:sparse_trace(x,k))
        for seed in SEEDS:
            if (k,seed) in done: print("skip",k,seed); continue
            rows=[r for r in rows if not (int(r["k"])==k and int(r["seed"])==seed)]
            def cb(model,step,loss,k=k,seed=seed,gold=gold,mode=mode):
                m=fx.evaluate(model,task,va,mode,ARGS,DEVICE,gold_trace_of=gold,desc=f"k={k}/s{seed}/t{step}")
                rows.append({"k":k,"depth":D,"seed":seed,"step":step,"loss":loss,"answer_accuracy":m["answer_accuracy"],"exact_stride_trace":None if k==D else m["exact_trace_accuracy"]}); pd.DataFrame(rows).to_csv(path,index=False); print(rows[-1])
            model,_=fx.train(task,tr,mode,seed,ARGS,DEVICE,split=split,target_of=target,checkpoints=CKPTS,at_checkpoint=cb,desc=f"stride k={k}/s{seed}"); del model; torch.cuda.empty_cache() if DEVICE.type=="cuda" else None
        del split
    df=pd.DataFrame(rows); summary=[]
    for (k,s),g in df.groupby(["k","seed"]):
        g=g.sort_values("step"); hit=g[g.answer_accuracy>=1/16+.05]; summary.append({"k":int(k),"seed":int(s),"leave_chance_step":None if hit.empty else int(hit.step.iloc[0]),"final_answer":float(g.answer_accuracy.iloc[-1]),"final_sparse_trace":g.exact_stride_trace.iloc[-1]})
    sdf=pd.DataFrame(summary); sdf.to_csv(OUT/"p4a_stride_transformer_summary.csv",index=False); display(sdf.groupby("k").agg(final_answer=("final_answer","mean"),leave_chance=("leave_chance_step","mean"))); return df,sdf


In [ ]:
stride_df,stride_summary=run_stride()

### Remaining matrix runs
`fastexec` does **not** apply to the matrix stride exponent or GD/AdamW escape robustness. Resume those in the existing matrix notebook/code; do not rerun completed CSV entries.